# AI-Generated Text Detection: Clustering (Unsupervised)

This notebook demonstrates how to implement an unsupervised clustering model (e.g., K-Means) to detect AI-generated text. Unlike supervised models (Logistic Regression, Random Forest, SVM), clustering algorithms do not use the labels during the training process. Instead, they find natural groupings in the data. 

We will evaluate if these natural groupings correspond to "Human" vs "AI" text.

In [ ]:
# Install Visualization and ML Libraries inline
%pip install matplotlib seaborn scikit-learn

In [ ]:
# ============================================================
# Import Libraries
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.manifold import TSNE

from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_auc_score, precision_score, recall_score
from sklearn.metrics import adjusted_rand_score, v_measure_score

import warnings
warnings.filterwarnings('ignore')

### Load the External Dataset

In [ ]:
# ============================================================
# Load Data - Automatically detects Colab vs Local
# ============================================================
import os

# Try different paths (Colab vs Local)
possible_paths = [
    "train_v4_drcat_01.csv",                              # Colab (uploaded file)
    "DAIGT-V4-TRAIN-DATASEt/train_v4_drcat_01.csv",       # Local (subfolder)
    "../DAIGT-V4-TRAIN-DATASEt/train_v4_drcat_01.csv",    # Local (parent folder)
]

data_path = None
for path in possible_paths:
    if os.path.exists(path):
        data_path = path
        break

if data_path is None:
    raise FileNotFoundError(
        "Could not find data file! Please either:\n"
        "1. Upload 'train_v4_drcat_01.csv' in Colab, or\n"
        "2. Make sure the DAIGT-V4 folder is in the correct location locally"
    )

print(f"Loading data from: {data_path}")
df = pd.read_csv(data_path)

# Display basic info
print(f"\nDataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

### Data Preprocessing and TF-IDF Vectorization

In [ ]:
# Check for missing values
if df['text'].isnull().sum() > 0:
    df.dropna(subset=['text'], inplace=True)

X = df['text']
y = df['label']

# ============================================================
# Train-Test Split
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    stratify=y, 
    random_state=42
)

print(f"Training instances: {len(X_train)}")
print(f"Testing instances: {len(X_test)}")

# ============================================================
# TF-IDF Vectorization
# ============================================================
vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1,2),
    min_df=3,
    max_df=0.9,
    stop_words='english'
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(f"TF-IDF feature shape: {X_train_tfidf.shape}")

### Dimensionality Reduction (Latent Semantic Analysis)

K-Means does not work well on high-dimensional sparse data like TF-IDF matrices due to the "curse of dimensionality". We use `TruncatedSVD` to reduce the dimensions, followed by `Normalizer`.

In [ ]:
# ============================================================
# LSA Pipeline: TruncatedSVD + Normalizer
# ============================================================
n_components = 100

# 1. SVD
svd = TruncatedSVD(n_components=n_components, random_state=42)
X_train_reduced = svd.fit_transform(X_train_tfidf)
X_test_reduced = svd.transform(X_test_tfidf)

print(f"Explained variance ratio: {svd.explained_variance_ratio_.sum():.3f}")

# 2. Normalization
normalizer = Normalizer(copy=False)
X_train_lsa = normalizer.fit_transform(X_train_reduced)
X_test_lsa = normalizer.transform(X_test_reduced)

print(f"LSA feature shape: {X_train_lsa.shape}")

### Train K-Means Clustering Model

In [ ]:
# ============================================================
# K-Means Clustering
# ============================================================
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
kmeans.fit(X_train_lsa)

# Predict clusters for train and test
train_clusters = kmeans.predict(X_train_lsa)
test_clusters = kmeans.predict(X_test_lsa)

### Cluster Mapping and Evaluation

Since the algorithm assigns arbitrary IDs (0 or 1) to the clusters, we must determine which cluster corresponds to the "Human" label and which to the "AI" label by mapping them to the majority true class.

In [ ]:
# ============================================================
# Map Clusters to True Labels
# ============================================================
def map_clusters(true_labels, predicted_clusters):
    # Convert true_labels to numpy array to prevent pandas index alignment issues with the 0-indexed predicted_clusters array
    df_map = pd.DataFrame({'true_label': np.array(true_labels), 'cluster': predicted_clusters})
    
    # Find the most frequent true label for each cluster
    cluster_0_label = df_map[df_map['cluster'] == 0]['true_label'].mode()[0]
    cluster_1_label = df_map[df_map['cluster'] == 1]['true_label'].mode()[0]
    
    # Create a mapping dictionary
    mapping = {0: cluster_0_label, 1: cluster_1_label}
    
    # If both clusters map to the same label, something is wrong, we default to 0 and 1
    if cluster_0_label == cluster_1_label:
        print("Warning: Both clusters mapped to the same class! Keeping default 0 and 1.")
        mapping = {0: 0, 1: 1}
        
    return mapping

cluster_mapping = map_clusters(y_train, train_clusters)
print(f"Mapping: Cluster 0 -> Label {cluster_mapping[0]}, Cluster 1 -> Label {cluster_mapping[1]}")

# Apply mapping to train and test sets
y_train_pred = np.array([cluster_mapping[c] for c in train_clusters])
y_test_pred = np.array([cluster_mapping[c] for c in test_clusters])

# ============================================================
# Train Baseline Logistic Regression for Comparison
# ============================================================
print("\nTraining Baseline Logistic Regression Model for comparison...")
baseline_lr = LogisticRegression(max_iter=1000, solver='lbfgs')
baseline_lr.fit(X_train_tfidf, y_train)
y_pred_lr = baseline_lr.predict(X_test_tfidf)
y_pred_proba_lr = baseline_lr.predict_proba(X_test_tfidf)[:, 1]

# Calculate LR Metrics
roc_auc_lr = roc_auc_score(y_test, y_pred_proba_lr)
precision_lr = precision_score(y_test, y_pred_lr)
recall_lr = recall_score(y_test, y_pred_lr)
f1_lr = f1_score(y_test, y_pred_lr)

# Calculate K-Means Metrics
roc_auc_km = roc_auc_score(y_test, y_test_pred) # Using binary predictions as pseudo-proba for AUC
precision_km = precision_score(y_test, y_test_pred)
recall_km = recall_score(y_test, y_test_pred)
f1_km = f1_score(y_test, y_test_pred)

# ============================================================
# Evaluation Comparison
# ============================================================
print("\nClustering Metrics (Train):")
print(f"Adjusted Rand Index: {adjusted_rand_score(y_train, train_clusters):.4f}")
print(f"V-Measure Score: {v_measure_score(y_train, train_clusters):.4f}")

metrics_df = pd.DataFrame({
    'Metric': ['ROC-AUC', 'Precision', 'Recall', 'F1-Score'],
    'K-Means (Unsupervised)': [roc_auc_km, precision_km, recall_km, f1_km],
    'Logistic Regression (Baseline)': [roc_auc_lr, precision_lr, recall_lr, f1_lr]
})

print("\n============================================================")
print("MODEL COMPARISON TABLE")
print("============================================================")
print(metrics_df.to_string(index=False, float_format="{:.4f}".format))
print("============================================================")

print("\nK-Means Classification Report (Test):")
print(classification_report(y_test, y_test_pred, target_names=['Human (0)', 'AI (1)']))

### Visualizing the Clusters

We use t-SNE to further reduce the LSA dimensions down to 2 for 2D plotting.

In [ ]:
# Sample a subset for faster t-SNE visualization
sample_size = 2000
if X_test_lsa.shape[0] > sample_size:
    indices = np.random.choice(X_test_lsa.shape[0], sample_size, replace=False)
    X_vis = X_test_lsa[indices]
    y_true_vis = y_test.iloc[indices].values
    y_pred_vis = y_test_pred[indices]
else:
    X_vis = X_test_lsa
    y_true_vis = y_test.values
    y_pred_vis = y_test_pred

# t-SNE reduction to 2D
tsne = TSNE(n_components=2, random_state=42)
X_2d = tsne.fit_transform(X_vis)

# Plotting
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: True Labels
scatter1 = ax1.scatter(X_2d[:, 0], X_2d[:, 1], c=y_true_vis, cmap='coolwarm', alpha=0.6, edgecolors='w', s=40)
ax1.set_title("True Labels (0: Human, 1: AI)")
ax1.set_xlabel("t-SNE Component 1")
ax1.set_ylabel("t-SNE Component 2")
fig.colorbar(scatter1, ax=ax1, ticks=[0, 1])

# Plot 2: Predicted Clusters
scatter2 = ax2.scatter(X_2d[:, 0], X_2d[:, 1], c=y_pred_vis, cmap='coolwarm', alpha=0.6, edgecolors='w', s=40)
ax2.set_title("K-Means Predicted Clusters")
ax2.set_xlabel("t-SNE Component 1")
ax2.set_ylabel("t-SNE Component 2")
fig.colorbar(scatter2, ax=ax2, ticks=[0, 1])

plt.tight_layout()
plt.show()